# Text-to-Speech with Tacotron2 and HifiGAN

This is an English female voice TTS demo using open source projects [BogiHsu/Tacotron2-PyTorch](https://github.com/BogiHsu/Tacotron2-PyTorch) and [jik876/hifi-gan](https://github.com/jik876/hifi-gan).

Please enable GPU acceleration in Colab before you start running the code.

## Set up environment

In [ ]:
!git clone https://github.com/BogiHsu/Tacotron2-PyTorch.git
!pip install -r Tacotron2-PyTorch/requirements.txt
!git clone https://github.com/jik876/hifi-gan.git
!mkdir -p mel_files

## Download Tacotron2 pretrained model

In [ ]:
!wget https://github.com/BogiHsu/Tacotron2-PyTorch/releases/download/lj-200k-b512/ckpt_200000 -O Tacotron2-PyTorch/ckpt_200000

## Download HifiGAN pretrained model

In [ ]:
import gdown
dl = {'hifi-gan/config.json': 'https://drive.google.com/u/1/uc?id=1aDh576AEYA5eTjhx7sew1qcCM_Y526jc&export=download',
      'hifi-gan/generator_v1': 'https://drive.google.com/u/1/uc?id=14NENd4equCBLyyCSke114Mv6YR_j_uFs&export=download'}
for k in dl:
    gdown.download(dl[k], k)

## Tacotron2 inference

In [ ]:
%cd Tacotron2-PyTorch
!python3 inference.py --ckpt_pth=ckpt_200000 --text='Tacotron is awesome.' --npy_pth=../mel_files/1
%cd ../

## HifiGAN inference

In [ ]:
%cd hifi-gan
!python3 inference_e2e.py --checkpoint_file generator_v1 --input_mels_dir ../mel_files --output_dir ../wav_files
%cd ../

## Play synthesized wav file

In [ ]:
import IPython
IPython.display.Audio('wav_files/1_generated_e2e.wav')

In [1]:
import IPython

In [18]:
IPython.display.Audio('wavdir/result_nepali.wav')

In [3]:
IPython.display.Audio('wavdir/result_nepali_2.wav')

In [8]:
%load_ext tensorboard

In [12]:
%tensorboard --logdir=logdir

In [13]:
!kill 2240

/bin/bash: line 0: kill: (2240) - No such process


In [27]:
import inference as riri_inference
import torch
import os
from scipy.io.wavfile import write

In [28]:
model = riri_inference.load_model('ckpt/ckpt_riri_24k_16bs_392431')

Initializing model with parameters: n_symbols=148, symbols_embedding_dim= 512, 


In [29]:
text = 'त्यस्तै कारोबारका लागि १३ लाख बढी लगानीकर्ताले अनलाइन सुविधा लिएका छन् . छन् छन् छन् छन् छन् .'

In [30]:
output = riri_inference.infer(text, model)

Terminated by gate.


In [31]:
npy = output[1]

In [32]:
npy.shape

torch.Size([1, 80, 573])

In [33]:
npy[0].shape

torch.Size([80, 573])

In [34]:
waveglow = torch.load("../waveglow/checkpoints/waveglow_24000")['model']

In [35]:
waveglow = waveglow.remove_weightnorm(waveglow)
waveglow.cuda().eval()

WaveGlow(
  (upsample): ConvTranspose1d(80, 80, kernel_size=(1024,), stride=(256,))
  (WN): ModuleList(
    (0): WN(
      (in_layers): ModuleList(
        (0): Conv1d(256, 512, kernel_size=(3,), stride=(1,), padding=(1,))
        (1): Conv1d(256, 512, kernel_size=(3,), stride=(1,), padding=(2,), dilation=(2,))
        (2): Conv1d(256, 512, kernel_size=(3,), stride=(1,), padding=(4,), dilation=(4,))
        (3): Conv1d(256, 512, kernel_size=(3,), stride=(1,), padding=(8,), dilation=(8,))
        (4): Conv1d(256, 512, kernel_size=(3,), stride=(1,), padding=(16,), dilation=(16,))
        (5): Conv1d(256, 512, kernel_size=(3,), stride=(1,), padding=(32,), dilation=(32,))
        (6): Conv1d(256, 512, kernel_size=(3,), stride=(1,), padding=(64,), dilation=(64,))
        (7): Conv1d(256, 512, kernel_size=(3,), stride=(1,), padding=(128,), dilation=(128,))
      )
      (res_skip_layers): ModuleList(
        (0): Conv1d(256, 512, kernel_size=(1,), stride=(1,))
        (1): Conv1d(256, 512, k

In [36]:
from apex import amp
waveglow, _ = amp.initialize(waveglow, [], opt_level="O3")

Selected optimization level O3:  Pure FP16 training.
Defaults for this optimization level are:
enabled                : True
opt_level              : O3
cast_model_type        : torch.float16
patch_torch_functions  : False
keep_batchnorm_fp32    : False
master_weights         : False
loss_scale             : 1.0
Processing user overrides (additional kwargs that are not None)...
After processing overrides, optimization options are:
enabled                : True
opt_level              : O3
cast_model_type        : torch.float16
patch_torch_functions  : False
keep_batchnorm_fp32    : False
master_weights         : False
loss_scale             : 1.0


In [37]:
# denoiser = Denoiser(waveglow).cuda()

In [38]:
mel = torch.autograd.Variable(npy.cuda())

In [39]:
sigma = 0.6
is_fp16 = True
# mel = torch.unsqueeze(mel, 0)
mel = mel.half() if is_fp16 else mel


In [40]:
mel, mel.shape

(tensor([[[ -9.5391,  -9.4766,  -7.4961,  ...,  -5.1445,  -5.0312,  -5.2734],
          [-11.5000, -11.3594,  -7.6406,  ...,  -5.9414,  -5.9375,  -6.1445],
          [-11.3281, -11.1641,  -7.3984,  ...,  -6.1289,  -6.0547,  -6.2539],
          ...,
          [-11.5156, -11.4844, -10.2031,  ...,  -9.6953,  -9.6172,  -9.7656],
          [-11.5078, -11.4766, -10.2188,  ...,  -9.8438,  -9.7266,  -9.8438],
          [-11.4609, -11.4297, -10.2266,  ...,  -9.8672,  -9.7500,  -9.8672]]],
        device='cuda:0', dtype=torch.float16),
 torch.Size([1, 80, 573]))

In [41]:
MAX_WAV_VALUE = 32768.0

In [42]:
with torch.no_grad():
    audio = waveglow.infer(mel, sigma=sigma)
    audio = audio * MAX_WAV_VALUE

In [43]:
output_dir = "./"
file_name = "result"
sampling_rate = 22050
audio = audio.squeeze()
audio = audio.cpu().numpy()
audio = audio.astype('int16')
audio_path = os.path.join(output_dir, "{}_synthesis.wav".format(file_name))
write(audio_path, sampling_rate, audio)